# Visualize inversion results

Use this notebook to visualize some standard inversion results.

See `utils.py` for the code to generate plots and count observations in the region of interest (imported below).

Notebook can be accessed via AWS following these instructions: https://docs.aws.amazon.com/dlami/latest/devguide/setup-jupyter.html)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib
import sys
import os
import glob
import re
import datetime
import pickle as pickle
import pandas as pd
import cartopy
import cartopy.crs as ccrs
import colorcet as cc
from shapely.geometry.polygon import Polygon
import shapely.ops as ops
from functools import partial
import pyproj

from utils import (
    plot_field,
    plot_field_gchp,
    load_obj,
    sum_total_emissions,
    count_obs_in_mask,
    get_posterior_emissions,
    get_mean_emissions,
    plot_ensemble,
    plot_hyperparameter_analysis,
    export_visualization_outputs,
    build_sector_weight_matrix,
    reduce_matrix_by_sector,
    covariance_to_correlation,
)

from config_utils import load_config

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

## Setup

In [ ]:
# Read the configuration file
config = load_config("/home/ubuntu/integrated_methane_inversion/config.yml")

# Save out the species as its own variable
species = config["Species"]

In [ ]:
if config['UseGCHP']:
    CSgridDir = f"{os.path.expandvars(config['OutputPath']) }/{config['RunName']}/CS_grids"
    gridfpath = f"{CSgridDir}/grids.c{config['CS_RES']}.nc"
    grid_ds = xr.open_dataset(gridfpath)

# Paths to prior emissions, inversion results, GEOS/satellite data, posterior simulation
# update prior prefix and period based on whether using kalman mode
if config["KalmanMode"]:
    prior_prefix = "./../.."
    period = int(os.getcwd().split("kf_inversions/period")[-1])  # get current period
    periods_df = pd.read_csv("./../../periods.csv")
    start_date = periods_df.iloc[period - 1, 0]
    end_date = periods_df.iloc[period - 1, 1]
    prior_sf_pth = f"{prior_prefix}/archive_sf/prior_sf_period{period}.nc"
else:
    prior_prefix = "./.."
    start_date = config["StartDate"]
    end_date = config["EndDate"]

prior_cache = f"{prior_prefix}/hemco_prior_emis/OutputDir/"
gc_fpath = f"./data_geoschem/GEOSChem.SpeciesConc.{start_date}_0000z.nc4"

satdat_dir = (
    "./data_converted_prior" if config["LognormalErrors"] else "./data_converted"
)
results_pth = (
    "./gridded_posterior_ln.nc"
    if config["LognormalErrors"]
    else "./gridded_posterior.nc"
)
inversion_result_path = (
    "./inversion_result_ln.nc" if config["LognormalErrors"] else "./inversion_result.nc"
)
posterior_dir = "./data_converted_posterior"
visualization_dir = (
    "./data_visualization_prior"
    if config["LognormalErrors"]
    else "./data_visualization"
)
posterior_viz_dir = "./data_visualization_posterior"

plot_save_path = ("./output" if not config["KalmanMode"] else f"../output/period_{period}_plots")

In [ ]:
# Open the state vector file
if config["KalmanMode"] and config["DynamicKFClustering"]:
    state_vector_filepath = f"./../../archive_sv/StateVector_{period}.nc"
else:
    state_vector_filepath = "./../StateVector.nc"

state_vector = xr.open_dataset(state_vector_filepath).squeeze()
state_vector_labels = state_vector["StateVector"]

# Identify the last element of the region of interest
last_ROI_element = int(state_vector_labels.isel(
    lat=slice(config["BufferRings"] + 4, -config["BufferRings"] - 4),
    lon=slice(config["BufferRings"] + 4, -config["BufferRings"] - 4)
).max())

# Define mask for region of interest
mask = (state_vector_labels <= last_ROI_element)

In [ ]:
# Set latitude/longitude bounds for plots
if not config['UseGCHP']:
    # Trim 0.5-2.5 degrees to remove GEOS-Chem buffer zone
    if config["Res"] == "0.125x0.15625":
        degx = 4 * 0.15625
        degy = 4 * 0.125
    elif config["Res"] == "0.25x0.3125":
        degx = 4 * 0.3125
        degy = 4 * 0.25
    elif config["Res"] == "0.5x0.625":
        degx = 4 * 0.625
        degy = 4 * 0.5
    elif config["Res"] == "2.0x2.5":
        degx = 4 * 2.5
        degy = 4 * 2.0

    lon_bounds = [
        np.min(state_vector.lon.values) + degx,
        np.max(state_vector.lon.values) - degx,
    ]
    lat_bounds = [
        np.min(state_vector.lat.values) + degy,
        np.max(state_vector.lat.values) - degy,
    ]
elif config['STRETCH_GRID']:
    buffer_bounds = 0.
    temp_lons = grid_ds['corner_lons'].values[5,...].copy()
    temp_lons[temp_lons>180] -= 360
    lon_min = max(temp_lons.min() - buffer_bounds, -180)
    lon_max = min(temp_lons.max() + buffer_bounds, 180)
    lat_min = max(grid_ds['corner_lats'].values[5,...].min(), -90)
    lat_max = min(grid_ds['corner_lats'].values[5,...].max(), 90)
    lon_bounds = [lon_min, lon_max]
    lat_bounds = [lat_min, lat_max]
else:
    lon_bounds = [-180, 180]
    lat_bounds = [-90, 90]

## State vector

In [ ]:
fig = plt.figure(figsize=(8, 8))
plt.rcParams.update({"font.size": 16})
ax = fig.subplots(1, 1, subplot_kw={"projection": ccrs.PlateCarree()})
num_colors = state_vector_labels.where(mask).max().item()
sv_cmap = matplotlib.colors.ListedColormap(np.random.rand(int(num_colors), 3))

if not config['UseGCHP']:
    plot_field(
        ax,
        state_vector_labels,
        cmap=sv_cmap,
        vmin=1,
        vmax=num_colors,
        title="State vector elements",
        cbar_label="Element Id",
        is_regional=config["isRegional"],
        save_path=plot_save_path
    )
else:
    plot_field_gchp(
        ax,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        state_vector_labels,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        cmap=sv_cmap,
        vmin=1,
        vmax=num_colors,
        title="State vector elements",
        cbar_label="Element Id",
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
        save_path=plot_save_path
    )

## Prior and posterior emissions in the region of interest

In [ ]:
# Prior emissions
prior_ds = get_mean_emissions(start_date, end_date, prior_cache)
if config["OptimizeSoil"]:
    prior = prior_ds[f"Emis{species}_Total"]
else:
    prior = prior_ds[f"Emis{species}_Total_ExclSoilAbs"]

if config["KalmanMode"]:
    # properly apply nudged sfs to prior in Kalman mode
    prior_sf = xr.open_dataset(prior_sf_pth)
    prior_ds = get_posterior_emissions(prior_ds, prior_sf, species, config["OptimizeSoil"])
    prior = prior_ds[f"Emis{species}_Total"]

# Optimized scale factors
scale_ds = xr.open_dataset(results_pth)
scale = scale_ds["ScaleFactor"]

# ensemble results
ens_scale_ds = xr.open_dataset(results_pth.replace(".nc", "_ensemble.nc"))
ens_inv_result = xr.open_dataset(inversion_result_path.replace(".nc", "_ensemble.nc"))

# Posterior emissions
posterior_ds = get_posterior_emissions(prior_ds, scale_ds, species, config["OptimizeSoil"])
if config["OptimizeSoil"]:
    posterior = posterior_ds[f"Emis{species}_Total"]
else:
    posterior = posterior_ds[f"Emis{species}_Total_ExclSoilAbs"]

In [ ]:
# Total emissions in the region of interest
if config['UseGCHP']:
    areas = grid_ds['area']
else: 
    areas = prior_ds["AREA"]

total_prior_emissions = sum_total_emissions(prior, areas, mask)
total_posterior_emissions = sum_total_emissions(posterior, areas, mask)

print("Prior         emissions :", total_prior_emissions, "Tg/y")
print(
    "Posterior     emissions :",
    total_posterior_emissions,
    "Tg/y (mean xhat)",
)

# Plot the ensemble of posterior emissions
# and calculate the mean and standard deviation
# Note: update the exclude_members list to exclude ensemble members from the analysis
# integer list of ensemble members to exclude from analysis
exclude_members = []  # eg. [0, 4, 5]
ensemble_members = ens_scale_ds.ensemble.values
ensemble_members = [num for num in ensemble_members if num not in exclude_members]
num_ensemble_members = len(ensemble_members)

if num_ensemble_members > 1:
    ensemble_posteriors = []
    for imem in ensemble_members:
        ensemble_posteriors.append(
            get_posterior_emissions(
                prior_ds, ens_scale_ds.sel(ensemble=imem)["ScaleFactor"], species, config["OptimizeSoil"]
            )
        )

    ens_totals_posterior = np.array(
        [
            sum_total_emissions(item[f"Emis{species}_Total_ExclSoilAbs"], areas, mask)
            for item in ensemble_posteriors
        ]
    )
    ens_totals_min = np.min(ens_totals_posterior)
    ens_totals_max = np.max(ens_totals_posterior)
    print("Ensemble range:", f" {ens_totals_min}-{ens_totals_max} Tg/yr")

    # Plot the totals
    fig = plt.figure(figsize=(8, 8))
    plt.rcParams.update({"font.size": 14})
    ax = fig.subplots(1, 1)
    plot_ensemble(
        ax,
        ens_totals_posterior,
        total_prior_emissions,
        plot_save_path=plot_save_path,
    )

In [ ]:
# Plot sensitivity of total emissions to hyperparameters
if num_ensemble_members > 1:
    # Collect parameters with multiple values used in ensemble members
    params = [
        "prior_err",
        "obs_err",
        "gamma",
        "prior_err_bc",
        "prior_err_oh",
    ]
    if config["LognormalErrors"]:
        params.append("prior_err_buffer")
    params_dict = {
        key: ens_inv_result[key].values[ensemble_members]
        for key in params
        if len(np.unique(ens_inv_result[key].values[ensemble_members])) > 1
    }

    # Only proceed if there are axes to plot
    num_axes = len(params_dict.keys())
    if num_axes > 0:
        fig, axs = plt.subplots(num_axes, 1, figsize=(8, 3 * num_axes), squeeze=False)
        plt.rcParams.update({"font.size": 14})
        plot_hyperparameter_analysis(axs, ens_totals_posterior, params_dict)

In [ ]:
# Plot prior emissions
fig = plt.figure(figsize=(8, 8))
plt.rcParams.update({"font.size": 16})
ax = fig.subplots(1, 1, subplot_kw={"projection": ccrs.PlateCarree()})

prior_kgkm2h = prior * (1000**2) * 60 * 60  # Units kg/km2/h

if config['UseGCHP']:
    plot_field_gchp(
        ax,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        prior_kgkm2h,
        cmap=cc.cm.linear_kryw_5_100_c67_r,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        vmin=0,
        vmax=14,
        title="Prior emissions",
        cbar_label="Emissions (kg km$^{-2}$ h$^{-1}$)",
        only_ROI=True,
        state_vector_labels=state_vector_labels,
        last_ROI_element=last_ROI_element,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
        save_path=plot_save_path
    )
else:
    plot_field(
        ax,
        prior_kgkm2h,
        cmap=cc.cm.linear_kryw_5_100_c67_r,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        vmin=0,
        vmax=14,
        title="Prior emissions",
        cbar_label="Emissions (kg km$^{-2}$ h$^{-1}$)",
        only_ROI=True,
        state_vector_labels=state_vector_labels,
        last_ROI_element=last_ROI_element,
        is_regional=config["isRegional"],
        save_path=plot_save_path
    )

In [ ]:
# Plot posterior emissions
fig = plt.figure(figsize=(8, 8))
plt.rcParams.update({"font.size": 16})
ax = fig.subplots(1, 1, subplot_kw={"projection": ccrs.PlateCarree()})

posterior_kgkm2h = posterior * (1000**2) * 60 * 60  # Units kg/km2/h

if config['UseGCHP']:
    plot_field_gchp(
        ax,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        posterior_kgkm2h,
        cmap=cc.cm.linear_kryw_5_100_c67_r,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        vmin=0,
        vmax=14,
        title="Posterior emissions",
        cbar_label="Emissions (kg km$^{-2}$ h$^{-1}$)",
        only_ROI=True,
        state_vector_labels=state_vector_labels,
        last_ROI_element=last_ROI_element,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
        save_path=plot_save_path
    )
else:
    plot_field(
        ax,
        posterior_kgkm2h,
        cmap=cc.cm.linear_kryw_5_100_c67_r,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        vmin=0,
        vmax=14,
        title="Posterior emissions",
        cbar_label="Emissions (kg km$^{-2}$ h$^{-1}$)",
        only_ROI=True,
        state_vector_labels=state_vector_labels,
        last_ROI_element=last_ROI_element,
        is_regional=config["isRegional"],
        save_path=plot_save_path
    )

## Sectoral Emissions

In [ ]:
# Extract sector names
if not config['OptimizeSoil']:
    sectors = [
        var
        for var in list(posterior_ds.keys())
        if "EmisCH4" in var and not ("Total" in var or "Excl" in var or "Soil" in var)
    ]
else:
    sectors = [
        var
        for var in list(posterior_ds.keys())
        if "EmisCH4" in var and not ("Total" in var or "Excl" in var)
    ]

# Calculate total emissions for each sector
prior_sector_vals = []
post_sector_vals = []
sector_names = []
for sector in sectors:
    post_val = sum_total_emissions(posterior_ds[sector], areas, mask)
    prior_val = sum_total_emissions(prior_ds[sector], areas, mask)
    if (abs(post_val)>0) or (abs(prior_val)>0):
        post_sector_vals.append(post_val)
        prior_sector_vals.append(prior_val)
        sector_names.append(sector.replace(f"Emis{species}_", ""))


# Combine the lists into tuples and sort them based on post_sector_vals
combined = list(zip(sector_names, prior_sector_vals, post_sector_vals))
combined_sorted = sorted(combined, key=lambda x: x[2])
sector_names, prior_sector_vals, post_sector_vals = zip(*combined_sorted)

# Plot bars for prior and posterior
fig = plt.figure(figsize=(10, 5))
ax = fig.subplots(1, 1)
bar_height = 0.35
ind = np.arange(len(sector_names))
bars2 = ax.barh(
    ind + bar_height / 2,
    post_sector_vals,
    bar_height,
    color="steelblue",
    label="Posterior Emissions",
)
bars1 = ax.barh(
    ind - bar_height / 2,
    prior_sector_vals,
    bar_height,
    color="goldenrod",
    label="Prior Emissions",
)

# Add labels, title, and legend
ax.set_xlabel("Emissions ($Tg\ a^{-1}$)")
ax.set_ylabel("Sector")
ax.set_title("Sectoral emissions")
ax.set_yticks(ind)
ax.set_yticklabels(sector_names)
ax.legend()

plt.savefig(f"{plot_save_path}/Sectoral_emissions.png", bbox_inches="tight")

## Scale factors

In [ ]:
fig = plt.figure(figsize=(8, 8))
plt.rcParams.update({"font.size": 16})
ax = fig.subplots(1, 1, subplot_kw={"projection": ccrs.PlateCarree()})

if config['UseGCHP']:
    plot_field_gchp(
        ax,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        scale,
        cmap="RdBu_r",
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        vmin=0,
        vmax=2,
        title="Scale factors",
        cbar_label="Scale factor",
        only_ROI=True,
        state_vector_labels=state_vector_labels,
        last_ROI_element=last_ROI_element,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
        save_path=plot_save_path
    )
else:
    plot_field(
        ax,
        scale,
        cmap="RdBu_r",
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        vmin=0,
        vmax=2,
        title="Scale factors",
        cbar_label="Scale factor",
        only_ROI=True,
        state_vector_labels=state_vector_labels,
        last_ROI_element=last_ROI_element,
        is_regional=config["isRegional"],
        save_path=plot_save_path
    )

## Averaging kernel sensitivities

In [ ]:
S_post_grid = xr.open_dataset(results_pth)["S_post"]
A_grid = xr.open_dataset(results_pth)["A"]
avkern_ROI = A_grid.where(state_vector_labels <= last_ROI_element)

In [ ]:
fig = plt.figure(figsize=(8, 8))
plt.rcParams.update({"font.size": 16})
ax = fig.subplots(1, 1, subplot_kw={"projection": ccrs.PlateCarree()})
vmin = np.nanpercentile(avkern_ROI, 5)
vmax = np.nanpercentile(avkern_ROI, 95)

if config['UseGCHP']:
    plot_field_gchp(
        ax,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        avkern_ROI,
        cmap=cc.cm.CET_L19,
        vmin=vmin,
        vmax=vmax,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="Averaging kernel sensitivities",
        cbar_label="Sensitivity",
        only_ROI=True,
        state_vector_labels=state_vector_labels,
        last_ROI_element=last_ROI_element,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
        save_path=plot_save_path
    )
else:
    plot_field(
        ax,
        avkern_ROI,
        cmap=cc.cm.CET_L19,
        vmin=vmin,
        vmax=vmax,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="Averaging kernel sensitivities",
        cbar_label="Sensitivity",
        only_ROI=True,
        state_vector_labels=state_vector_labels,
        last_ROI_element=last_ROI_element,
        is_regional=config["isRegional"],
        save_path=plot_save_path
    )

In [ ]:
# ungridded inversion result is used to calculate DOFS using the trace of the averaging kernel
A_ROI = xr.open_dataset(inversion_result_path)["A"].values[:last_ROI_element, :last_ROI_element]
DOFS = np.trace(A_ROI)
print("DOFS =", DOFS)

### ROI sector-reduced averaging kernel and error correlations


In [ ]:
sector_weights_ROI = build_sector_weight_matrix(
    prior_ds,
    state_vector_labels,
    last_ROI_element,
    areas=areas,
    species=species,
    include_soil=config["OptimizeSoil"],
)

with xr.open_dataset(inversion_result_path) as inv_result:
    S_post_ROI = inv_result["S_post"].values[:last_ROI_element, :last_ROI_element]

sector_A_ROI = reduce_matrix_by_sector(A_ROI, sector_weights_ROI, use_pseudoinverse=True)
sector_S_post_ROI = reduce_matrix_by_sector(S_post_ROI, sector_weights_ROI)
sector_error_corr_ROI = pd.DataFrame(
    covariance_to_correlation(sector_S_post_ROI),
    index=sector_S_post_ROI.index,
    columns=sector_S_post_ROI.columns,
)

print("ROI sector-reduced diagnostics:", ", ".join(sector_weights_ROI.index))


In [ ]:
fig = plt.figure(figsize=(8, 7))
plt.rcParams.update({"font.size": 12})
ax = fig.subplots(1, 1)
im = ax.imshow(sector_error_corr_ROI, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("ROI sector-reduced posterior error correlations")
ax.set_xticks(np.arange(len(sector_error_corr_ROI.columns)))
ax.set_xticklabels(sector_error_corr_ROI.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(len(sector_error_corr_ROI.index)))
ax.set_yticklabels(sector_error_corr_ROI.index)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Correlation")
fig.tight_layout()
plt.savefig(f"{plot_save_path}/ROI_sector_reduced_error_correlation.png", bbox_inches="tight")


In [ ]:
# Sector color palette (consistent across runs)
_sector_palette = {
    "Wetlands": "#6FA8DC",
    "Livestock": "#8CCB7F",
    "Waste": "#F28E2B",
    "Oil/Gas": "#B07AA1",
    "Coal": "#7F7F7F",
    "Reservoirs": "#2CB1C9",
    "Rice": "#BCBD22",
    "Other": "#9E9E9E",
    "BiomassBurn": "#E15759",
    "Seeps": "#76B7B2",
    "Termites": "#EDC948",
    "Wastewater": "#59A14F",
    "Landfills": "#FF9DA7",
    "OtherAnth": "#9E9E9E",
}

sector_order = list(sector_A_ROI.index)
ncols = 2
nrows = int(np.ceil(len(sector_order) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3 * nrows), sharex=True, sharey=True)
plt.rcParams.update({"font.size": 14})
axes = np.atleast_1d(axes).ravel()
x = np.arange(len(sector_order))

# Compute shared y-limits; always include y=1 in range
all_vals = np.array([sector_A_ROI.loc[s, sector_order].to_numpy(dtype=float) for s in sector_order])
ymin = min(float(all_vals.min()), -0.05)
ymax = max(float(all_vals.max()) * 1.05, 1.1)

for i, sector in enumerate(sector_order):
    ax = axes[i]
    y = sector_A_ROI.loc[sector, sector_order].to_numpy(dtype=float)
    row_color = _sector_palette.get(sector, "0.35")
    ax.axhline(0, color="0.45", linestyle="--", linewidth=1.5)
    # reference line at 1 (diagonal upper bound)
    ax.axhline(1, color="0.65", linestyle=":", linewidth=1.2)
    ax.plot(
        x, y,
        color=row_color,
        linewidth=2.5,
        marker="o",
        markersize=6,
        markeredgecolor="white",
        markeredgewidth=0.8,
    )
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(-0.5, len(sector_order) - 0.5)
    ax.set_title(sector, color=row_color, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", width=1.2, length=4)
    if i % ncols == 0:
        ax.set_ylabel("AK sensitivity")

for ax in axes[len(sector_order):]:
    ax.axis("off")

# x-tick labels only on bottom row subplots
for ax in axes[(nrows - 1) * ncols:]:
    if ax.has_data():
        ax.set_xticks(x)
        ax.set_xticklabels(sector_order, rotation=45, ha="right")
for ax in axes[:(nrows - 1) * ncols]:
    ax.set_xticks(x)
    ax.tick_params(labelbottom=False)

fig.suptitle("ROI sector-reduced averaging kernel sensitivities by row", fontsize=16)
fig.tight_layout()
plt.savefig(f"{plot_save_path}/ROI_sector_reduced_averaging_kernel.png", bbox_inches="tight")


## Open satellite and GEOS-Chem columns

In [ ]:
# Get observed and GEOS-Chem-simulated satellite columns
def aggregate_data(data_files, data_posterior_files, get_GCindex=False):
    lat = np.array([])
    lon = np.array([])
    satellite = np.array([])
    geos_prior = np.array([])
    geos_posterior = np.array([])
    observation_count = np.array([])
    if get_GCindex:
        GC_index = np.array([], dtype=int)
    for i in range(len(data_files)):
        # Get paths
        pth = data_files[i]
        pth_posterior = data_posterior_files[i]
        # Load satellite/GEOS-Chem and Jacobian matrix data from the .pkl file
        obj = load_obj(pth)
        # If there is no corresponding file in data_posterior dir, skip
        if not os.path.isfile(pth_posterior):
            continue
        obj_posterior = load_obj(pth_posterior)
        # If there aren't any satellite observations on this day, skip
        if obj["obs_GC"].shape[0] == 0:
            continue
        # Otherwise, grab the satellite/GEOS-Chem data
        obs_GC = obj["obs_GC"]
        obs_GC_posterior = obj_posterior["obs_GC"]
        # Only consider data within latitude and longitude bounds
        ind = np.where(
            (obs_GC[:, 2] >= lon_bounds[0])
            & (obs_GC[:, 2] <= lon_bounds[1])
            & (obs_GC[:, 3] >= lat_bounds[0])
            & (obs_GC[:, 3] <= lat_bounds[1])
        )
        if len(ind[0]) == 0:  # Skip if no data in bounds
            continue
        obs_GC = obs_GC[ind[0], :]  # satellite and GEOS-Chem data within bounds
        obs_GC_posterior = obs_GC_posterior[ind[0], :]
        # Record lat, lon, satellite mixing ratio, and geos mixing ratio
        lat = np.concatenate((lat, obs_GC[:, 3]))
        lon = np.concatenate((lon, obs_GC[:, 2]))
        satellite = np.concatenate((satellite, obs_GC[:, 0]))
        geos_prior = np.concatenate((geos_prior, obs_GC[:, 1]))
        observation_count = np.concatenate((observation_count, obs_GC[:, 4]))
        geos_posterior = np.concatenate((geos_posterior, obs_GC_posterior[:, 1]))

        if get_GCindex:
            GC_index = np.concatenate((GC_index, obj['GC_index'][ind[0]]))
    df = pd.DataFrame()
    df["lat"] = lat
    df["lon"] = lon
    df["satellite"] = satellite
    df["geos_prior"] = geos_prior
    df["geos_posterior"] = geos_posterior
    df["diff_satellite_prior"] = geos_prior - satellite
    df["diff_satellite_posterior"] = geos_posterior - satellite
    df["observation_count"] = observation_count
    if get_GCindex:
        df['GC_index'] = GC_index
    # Sort by latitude and longitude from northwest
    df = df.sort_values(by=["lat", "lon"], ascending=[False, True]).reset_index(drop=True)
    
    return df

def grid_superobs(config, superobs_df, gc_fpath, state_vector_filepath):
    varlist = ['satellite', 'geos_prior', 'geos_posterior', 'observation_count']
    long_name = ['Satellite observed methane concentrations',
                 'Methane concentrations from a prior simulation', 
                 'Methane concentrations from a posterior simulation',
                 'Number of observations in each simulation grid box']
    units = ['ppb', 'ppb', 'ppb', '1']
    
    def weighted_avg_column(df, val_col):
        return lambda x: np.average(df.loc[x.index, val_col], weights=df.loc[x.index, 'observation_count'])

    grouped = (
        superobs_df
        .groupby('GC_index', as_index=False)
        .agg({
            'observation_count': 'sum',
            'satellite': weighted_avg_column(superobs_df, 'satellite'),
            'geos_prior': weighted_avg_column(superobs_df, 'geos_prior'),
            'geos_posterior': weighted_avg_column(superobs_df, 'geos_posterior'),
        })
    )

    grouped['GC_index'] = grouped['GC_index'].astype(int)

    n_obs = len(grouped)

    # Read grid file
    GCds = xr.open_dataset(gc_fpath).squeeze()
    if config['UseGCHP']:
        lats = GCds['lats'].values
        lons = GCds['lons'].values
        GC_shape = lats.shape
    else:
        lat = GCds['lat'].values
        lon = GCds['lon'].values
        GC_shape = (len(lat), len(lon))

    var_values = np.full((len(varlist), *GC_shape), np.nan)

    for k in range(n_obs):
        gc_idx = grouped.loc[k, 'GC_index']
        if config['UseGCHP']:
            f, j, i = np.unravel_index(gc_idx, GC_shape)
            for vari in range(len(varlist)):
                var_values[vari, f, j, i] = grouped.loc[k, varlist[vari]]
        else:
            j, i = np.unravel_index(gc_idx, GC_shape)
            for vari in range(len(varlist)):
                var_values[vari, j, i] = grouped.loc[k, varlist[vari]]

    superobs_ds = xr.Dataset()

    if config['UseGCHP']:
        for vari in range(len(varlist)):
            superobs_ds[varlist[vari]] = xr.DataArray(
                var_values[vari, ...], dims=['nf', 'Ydim', 'Xdim'],
                coords=dict(lats=(['nf', 'Ydim', 'Xdim'], lats),
                            lons=(['nf', 'Ydim', 'Xdim'], lons)),
                attrs=dict(units=units[vari], long_name=long_name[vari])
            )

        superobs_ds['corner_lons'] = GCds['corner_lons']
        superobs_ds['corner_lats'] = GCds['corner_lats']
        superobs_ds.lats.attrs.update(units="degrees_north", long_name="Latitude")
        superobs_ds.lons.attrs.update(units="degrees_east", long_name="Longitude")

    else:
        for vari in range(len(varlist)):
            superobs_ds[varlist[vari]] = xr.DataArray(
                var_values[vari, ...], dims=['lat', 'lon'], coords=[lat, lon],
                attrs=dict(units=units[vari], long_name=long_name[vari])
            )

        superobs_ds.lat.attrs.update(units="degrees_north", long_name="Latitude")
        superobs_ds.lon.attrs.update(units="degrees_east", long_name="Longitude")

        # Subset to match statevector domain
        statevector_ds = xr.open_dataset(state_vector_filepath).squeeze()
        lon_range = [np.nanmin(statevector_ds['lon']), np.nanmax(statevector_ds['lon'])]
        lat_range = [np.nanmin(statevector_ds['lat']), np.nanmax(statevector_ds['lat'])]
        superobs_ds = superobs_ds.sel(
            lon=slice(lon_range[0], lon_range[1]),
            lat=slice(lat_range[0], lat_range[1])
        )

    superobs_ds['diff_satellite_prior'] = superobs_ds['geos_prior'] - superobs_ds['satellite']
    superobs_ds['diff_satellite_posterior'] = superobs_ds['geos_posterior'] - superobs_ds['satellite']
    # Add global attributes
    superobs_ds.attrs = {
        'Start date': config['StartDate'],
        'End date': config['EndDate']
    }

    return superobs_ds

from pathlib import Path
satellite_product = config["SatelliteProduct"]
# TODO: this function is copied and Frankenstein-ed from src/inversion_scripts/satellite_products/products.py
#       Eventually we should figure out how to reuse code instead of copying it here
#       For now when you add a new satellite product, add its observation date extraction here :)
def observation_date(path: str | Path) -> datetime.datetime:
    if satellite_product == "MSAT":
        """Extract the acquisition date from a MethaneSAT L3 filename."""
        name = Path(path).name
        match = re.search(r"(?:^|_)MSAT_L3_.*?_(\d{8})T\d{6}Z", name)
        if match is None:
            raise ValueError(
                f"Filename does not match the MSAT naming convention: {name!r}"
            )
        try:
            return datetime.datetime.strptime(match.group(1), "%Y%m%d")
        except ValueError as exc:
            raise ValueError(
                f"Invalid MSAT acquisition date in {name!r}: {match.group(1)}"
            ) from exc
    elif satellite_product in ("TROPOMI", "BlendedTROPOMI"): 
        """Extract the start of the TROPOMI acquisition window."""
        name = Path(path).name
        modes = "|".join(re.escape(mode) for mode in self.processing_modes)
        match = re.match(
            rf"^S5P_(?:{modes})_L2__CH4____(\d{{8}})T\d{{6}}", name
        )
        if match is None:
            raise ValueError(
                f"Filename does not match the {self.name} naming convention: {name!r}"
            )
        try:
            return datetime.datetime.strptime(match.group(1), "%Y%m%d")
        except ValueError as exc:
            raise ValueError(
                f"Invalid {self.name} acquisition date in {name!r}: {match.group(1)}"
            ) from exc
    else:
        raise ValueError(f"Satellite product {satellite_product} is not recognized")
    

gc_startdate = np.datetime64(datetime.datetime.strptime(str(config['StartDate']), "%Y%m%d"))
gc_enddate = np.datetime64(datetime.datetime.strptime(str(config['EndDate']), "%Y%m%d"))
allfiles = glob.glob(f"{satdat_dir}/*.pkl")
satdat_files = []
for index in range(len(allfiles)):
    filename = allfiles[index]
    strdate = observation_date(filename)
    if (strdate >= gc_startdate) and (strdate < gc_enddate):
        satdat_files.append(filename)
satdat_files.sort()

allfiles = glob.glob(f"{posterior_dir}/*.pkl")
posterior_files = []
for index in range(len(allfiles)):
    filename = allfiles[index]
    strdate = observation_date(filename)
    if (strdate >= gc_startdate) and (strdate < gc_enddate):
        posterior_files.append(filename)
posterior_files.sort()

allfiles = glob.glob(f"{visualization_dir}/*.pkl")
visualization_files = []
for index in range(len(allfiles)):
    filename = allfiles[index]
    strdate = observation_date(filename)
    if (strdate >= gc_startdate) and (strdate < gc_enddate):
        visualization_files.append(filename)
visualization_files.sort()

allfiles = glob.glob(f"{posterior_viz_dir}/*.pkl")
posterior_viz_files = []
for index in range(len(allfiles)):
    filename = allfiles[index]
    strdate = observation_date(filename)
    if (strdate >= gc_startdate) and (strdate < gc_enddate):
        posterior_viz_files.append(filename)
posterior_viz_files.sort()

superobs_df = aggregate_data(satdat_files, posterior_files, get_GCindex=True)
superobs_ds = grid_superobs(config, superobs_df, gc_fpath, state_vector_filepath)
netcdf_path = f"{plot_save_path}/netCDF"
os.makedirs(netcdf_path, exist_ok=True)  # Create the directory if it doesn't exist
superobs_fpathout = f'{netcdf_path}/super_observations_GC.nc'
superobs_ds.to_netcdf(superobs_fpathout, mode='w')

visualization_df = aggregate_data(visualization_files, posterior_viz_files)

n_obs = len(superobs_df["satellite"])

print(
    f'Found {n_obs} super-observations in the domain, representing {np.sum(superobs_df["observation_count"]).round(0)} satellite observations.'
)
superobs_df.head(20)

Note: This observation count is for the lat/lon bounds defined in input cell 5.

In [ ]:
# calculate some statistics for the prior
prior_std = np.round(np.std(superobs_df["diff_satellite_prior"]), 2)
prior_bias = np.round(
    np.average(
        superobs_df["diff_satellite_prior"], weights=superobs_df["observation_count"]
    ),
    2,
)
prior_RMSE = np.round(
    np.sqrt(
        np.average(
            superobs_df["diff_satellite_prior"] ** 2,
            weights=superobs_df["observation_count"],
        )
    ),
    2,
)

# and the posterior
posterior_std = np.round(np.std(superobs_df["diff_satellite_posterior"]), 2)
posterior_bias = np.round(
    np.average(
        superobs_df["diff_satellite_posterior"], weights=superobs_df["observation_count"]
    ),
    2,
)
posterior_RMSE = np.round(
    np.sqrt(
        np.average(
            superobs_df["diff_satellite_posterior"] ** 2,
            weights=superobs_df["observation_count"],
        )
    ),
    2,
)

# Print some error statistics
print(f"Bias in prior     : {prior_bias} +/- {prior_std} ppb")
print(f"RMSE prior        : {prior_RMSE} ppb")
print(f"Bias in posterior : {posterior_bias} +/- {posterior_std} ppb")
print(f"RMSE posterior    : {posterior_RMSE} ppb")

## Count observations within the region of interest

In [ ]:
print("Found", count_obs_in_mask(mask, superobs_df, config['UseGCHP']), "super-observations within the region of interest")

In [ ]:
# Data-driven colorbar range; this is unrelated to the spatial grid resolution.
def dynamic_range(vals):
    finite_vals = np.asarray(vals)[np.isfinite(vals)]
    if finite_vals.size == 0:
        raise ValueError("Cannot determine a colorbar range from empty data")
    vmin, vmax = np.nanpercentile(finite_vals, [5, 95])
    if vmin == vmax:
        vmin, vmax = vmin - 1, vmax + 1
    return vmin, vmax

## Compare satellite and GEOS-Chem columns

### Comparison at simulation resolution

In [ ]:
# Plot XCH4 from Satellite superobservations and GC 
ds = superobs_ds.copy()
# Mean satellite XCH4 columns on simulation grid
fig = plt.figure(figsize=(8, 8))
ax = fig.subplots(1, 1, subplot_kw={"projection": ccrs.PlateCarree()})

xch4_min, xch4_max = dynamic_range(ds["satellite"].values)

if config['UseGCHP']:
    plot_field_gchp(
        ax,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        ds["satellite"],
        cmap="Spectral_r",
        vmin=xch4_min,
        vmax=xch4_max,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="Satellite $X_{CH4}$",
        clean_title="Satellite XCH4 on GC grid",
        cbar_label="Column mixing ratio (ppb)",
        only_ROI=False,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
        save_path=plot_save_path
    )
else:
    plot_field(
        ax,
        ds["satellite"],
        cmap="Spectral_r",
        vmin=xch4_min,
        vmax=xch4_max,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="Satellite $X_{CH4}$",
        clean_title="Satellite XCH4 on GC grid",
        cbar_label="Column mixing ratio (ppb)",
        mask=mask,
        UseGCHP=config['UseGCHP'],
        only_ROI=False,
        is_regional=config["isRegional"],
        save_path=plot_save_path
    )

In [ ]:
# Mean prior and posterior GEOS-Chem XCH4 columns on GC grid
fig = plt.figure(figsize=(25, 8))
ax1, ax2 = fig.subplots(1, 2, subplot_kw={"projection": ccrs.PlateCarree()})

if config['UseGCHP']:
    plot_field_gchp(
        ax1,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        ds["geos_prior"],
        cmap="Spectral_r",
        vmin=xch4_min,
        vmax=xch4_max,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="GEOS-Chem $X_{CH4}$ (prior simulation)",
        clean_title="GEOS-Chem XCH4 (prior) on GC grid",
        cbar_label="Dry column mixing ratio (ppb)",
        only_ROI=False,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
    )

    plot_field_gchp(
        ax2,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        ds["geos_posterior"],
        cmap="Spectral_r",
        vmin=xch4_min,
        vmax=xch4_max,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="GEOS-Chem $X_{CH4}$ (posterior simulation)",
        clean_title="GEOS-Chem XCH4 Prior_and_Posterior on GC grid",
        cbar_label="Dry column mixing ratio (ppb)",
        only_ROI=False,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
        save_path=plot_save_path
    )
else:
    plot_field(
        ax1,
        ds["geos_prior"],
        cmap="Spectral_r",
        vmin=xch4_min,
        vmax=xch4_max,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="GEOS-Chem $X_{CH4}$ (prior simulation)",
        clean_title="GEOS-Chem XCH4 (prior) on GC grid",
        cbar_label="Dry column mixing ratio (ppb)",
        mask=mask,
        UseGCHP=config['UseGCHP'],
        only_ROI=False,
        is_regional=config["isRegional"],
    )

    plot_field(
        ax2,
        ds["geos_posterior"],
        cmap="Spectral_r",
        vmin=xch4_min,
        vmax=xch4_max,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="GEOS-Chem $X_{CH4}$ (posterior simulation)",
        clean_title="GEOS-Chem XCH4 Prior_and_Posterior on GC grid",
        cbar_label="Dry column mixing ratio (ppb)",
        mask=mask,
        UseGCHP=config['UseGCHP'],
        only_ROI=False,
        is_regional=config["isRegional"],
        save_path=plot_save_path
    )

In [ ]:
# Plot differences between GEOS-Chem and TROPOMI XCH4 on GC grid
fig = plt.figure(figsize=(25, 8))
ax1, ax2 = fig.subplots(1, 2, subplot_kw={"projection": ccrs.PlateCarree()})

if config['UseGCHP']:
    plot_field_gchp(
        ax1,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        ds["diff_satellite_prior"],
        cmap="RdBu_r",
        vmin=-40,
        vmax=40,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="Prior $-$ Satellite",
        clean_title="Prior-Satellite on GC grid",
        cbar_label="ppb",
        only_ROI=False,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
    )

    plot_field_gchp(
        ax2,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        ds["diff_satellite_posterior"],
        cmap="RdBu_r",
        vmin=-40,
        vmax=40,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="Posterior $-$ Satellite",
        clean_title="Prior_and_Posterior - Satellite on GC grid",
        cbar_label="ppb",
        only_ROI=False,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
        save_path=plot_save_path
    )
else:
    plot_field(
        ax1,
        ds["diff_satellite_prior"],
        cmap="RdBu_r",
        vmin=-40,
        vmax=40,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="Prior $-$ Satellite",
        clean_title="Prior-Satellite on GC grid",
        cbar_label="ppb",
        mask=mask,
        UseGCHP=config['UseGCHP'],
        only_ROI=False,
        is_regional=config["isRegional"],
    )

    plot_field(
        ax2,
        ds["diff_satellite_posterior"],
        cmap="RdBu_r",
        vmin=-40,
        vmax=40,
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="Posterior $-$ Satellite",
        clean_title="Prior_and_Posterior - Satellite on GC grid",
        cbar_label="ppb",
        mask=mask,
        UseGCHP=config['UseGCHP'],
        only_ROI=False,
        is_regional=config["isRegional"],
        save_path=plot_save_path
    )

In [ ]:
# Plot differences between posterior and prior simulated XCH4 on GC grid
fig = plt.figure(figsize=(8, 8))
ax = fig.subplots(1, 1, subplot_kw={"projection": ccrs.PlateCarree()})

ds['geos_diff'] = ds["geos_posterior"] - ds["geos_prior"]

if config['UseGCHP']:
    plot_field_gchp(
        ax,
        grid_ds['corner_lons'],
        grid_ds['corner_lats'],
        ds['geos_diff'],
        cmap="seismic",
        vmin=-np.nanmax(ds['geos_diff']),
        vmax=np.nanmax(ds['geos_diff']),
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="$\Delta X_{CH4}$ (Posterior $-$ Prior)",
        clean_title="Delta XCH4 Posterior-Prior on GC grid",
        cbar_label="ppb",
        only_ROI=False,
        is_regional=config["isRegional"],
        stretch_grid=config['STRETCH_GRID'],
        save_path=plot_save_path
    )
else:
    plot_field(
        ax,
        ds['geos_diff'],
        cmap="seismic",
        vmin=-np.nanmax(ds['geos_diff']),
        vmax=np.nanmax(ds['geos_diff']),
        lon_bounds=lon_bounds,
        lat_bounds=lat_bounds,
        title="$\Delta X_{CH4}$ (Posterior $-$ Prior)",
        clean_title="Delta XCH4 Posterior-Prior on GC grid",
        cbar_label="ppb",
        mask=mask,
        UseGCHP=config['UseGCHP'],
        only_ROI=False,
        is_regional=config["isRegional"],
        save_path=plot_save_path
    )

### Comparison at native satellite resolution

In [ ]:
# Plot the unrounded satellite observations at their native observation locations.
os.makedirs(plot_save_path, exist_ok=True)
native_df = visualization_df.dropna(subset=["lat", "lon"]).copy()
if native_df.empty:
    raise ValueError("No native satellite observations are available in the selected domain")

native_values = np.concatenate([
    native_df["satellite"].to_numpy(),
    native_df["geos_prior"].to_numpy(),
    native_df["geos_posterior"].to_numpy(),
])
native_vmin, native_vmax = dynamic_range(native_values)

def plot_native_field(ax, df, column, title, cmap, vmin, vmax, cbar_label):
    if config["isRegional"]:
        ax.add_feature(cartopy.feature.BORDERS, facecolor="none")
        ax.add_feature(cartopy.feature.STATES, facecolor="none", edgecolor="0.5")
    ax.coastlines(resolution="110m")
    values = df[column].to_numpy()
    scatter = ax.scatter(
        df["lon"], df["lat"], c=values, s=8, cmap=cmap,
        vmin=vmin, vmax=vmax, transform=ccrs.PlateCarree(),
    )
    ax.set_extent([*lon_bounds, *lat_bounds], crs=ccrs.PlateCarree())
    gridlines = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, alpha=0)
    gridlines.right_labels = False
    gridlines.top_labels = False
    ax.set_title(title)
    ax.get_figure().colorbar(
        scatter, ax=ax, label=cbar_label, fraction=0.041, pad=0.04
    )

fig, axes = plt.subplots(3, 1, figsize=(8, 18),
    subplot_kw={"projection": ccrs.PlateCarree()})
for ax, column, label in zip(
    axes,
    ["satellite", "geos_prior", "geos_posterior"],
    ["Satellite", "GEOS-Chem prior", "GEOS-Chem posterior"],
):
    plot_native_field(
        ax, native_df, column,
        f"{label} at native {satellite_product} observation locations",
        "Spectral_r", native_vmin, native_vmax, "Column mixing ratio (ppb)",
    )
fig.tight_layout()
fig.savefig(
    os.path.join(plot_save_path, f"native_{satellite_product}_xch4.png"),
    bbox_inches="tight",
)

In [ ]:
# Plot native-resolution prior/satellite and posterior/satellite differences.
native_diff_limit = np.nanmax(np.abs(native_df[[
    "diff_satellite_prior", "diff_satellite_posterior"
]].to_numpy()))

fig, axes = plt.subplots(2, 1, figsize=(8, 12),
    subplot_kw={"projection": ccrs.PlateCarree()})
for ax, column, label in zip(
    axes,
    ["diff_satellite_prior", "diff_satellite_posterior"],
    ["Prior $-$ satellite", "Posterior $-$ satellite"],
):
    plot_native_field(
        ax, native_df, column,
        f"{label} at native {satellite_product} observation locations",
        "RdBu_r", -native_diff_limit, native_diff_limit, "ppb",
    )
fig.tight_layout()
fig.savefig(
    os.path.join(plot_save_path, f"native_{satellite_product}_differences.png"),
    bbox_inches="tight",
)

In [ ]:
# Export visualization outputs
_ = export_visualization_outputs(
    plot_save_path=plot_save_path,
    species=species,
    start_date=start_date,
    end_date=end_date,
    total_prior_emissions=total_prior_emissions,
    total_posterior_emissions=total_posterior_emissions,
    DOFS=DOFS,
    prior_bias=prior_bias,
    posterior_bias=posterior_bias,
    prior_std=prior_std,
    posterior_std=posterior_std,
    prior_RMSE=prior_RMSE,
    posterior_RMSE=posterior_RMSE,
    df_for_count=superobs_df,
    mask=mask,
    combined_sorted=combined_sorted,
    prior_ds=prior_ds,
    posterior_ds=posterior_ds,
    areas=areas,
    num_ensemble_members=num_ensemble_members,
    ens_totals_posterior=(ens_totals_posterior if num_ensemble_members > 1 else None),
    ens_inv_result=(ens_inv_result if num_ensemble_members > 1 else None),
    ens_scale_ds=(ens_scale_ds if num_ensemble_members > 1 else None),
    state_vector_labels=state_vector_labels,
    scale=scale,
    avkern_ROI=avkern_ROI,
    ds_obs=superobs_ds,
    ds_obs_regridded=None,
)


----